In [ ]:
import pathlib
import re

In [ ]:
import mywidgets.Tools.uiStyle.MarkupRe as MarkupRe
import excelxml
from excelxml import OXLLoader, regex_range, unescape, excel_col_to_int, wscell_address, offset_rng, wscell_pattern

In [ ]:
from collections.abc import Callable
from typing import Any

In [ ]:
def exec_time(
    f1: Callable[..., Any],
    f2: Callable[..., Any],
    *args: Any,
    **kwargs: Any,
) -> None:
    '''
    Helper function that benchmarks two functions with the same arguments
    '''

    from statistics import median
    from timeit import repeat

    number = 1_000
    ntimes = 7

    # Check they return equivalent results before timing.
    assert f1(*args, **kwargs) == f2(*args, **kwargs)

    for fnc in (f1, f2):
        f_times = repeat(lambda f=fnc: f(*args, **kwargs), number=number, repeat=ntimes)
        print(f"{fnc.__name__}:    {median(f_times) / number * 1_000_000:.2f} µs/call")


In [ ]:
fpath = pathlib.Path(r'C:\Users\agmontesb\Documents\DIAN\Renta2025\F2517_AG2025')
fname = 'Reporte_Conciliación_Fiscal_F2517V8_AG2025_v1.0.0-2026.xlsm'
full_path = fpath / fname


In [ ]:
loader = OXLLoader(full_path)

In [ ]:
fout = r'C:\Users\agmontesb\Documents\GitHub\excel\tests\files\h3_eri_renta_liquida.xml'
# with open(fout, 'w') as f:
#     f.write(h3_xml)

In [ ]:
from excel_workbook import ExcelWorkbook, ExcelWorksheet, ExcelTable, pythonize_fml

In [ ]:
wb = excelxml.WorkBookXml(full_path)
ws = wb.active
ws.title
# wb.load_wb_data(full_path)


In [ ]:
nrow = 378
row = ws[f'{nrow}:{nrow}']
row

In [ ]:
ws.cell(378, 1, 378, 14)

In [ ]:
list(ws.iter_rows(378, 1, 378, 14))

In [ ]:
exec_time(ws.cell_new, ws.cell, 378, 1, 378, 14)

In [ ]:
fnc = lambda x: '{0: >4s}{1: >4s}'.format(*excelxml.wscell_address(x))

In [ ]:
gnc = lambda x: any(map(x.startswith, tuple('GHIJKLMN')))
gnc('N23')

In [ ]:
ws_range = 'H9:N671'
cells = ws[ws_range]

In [ ]:
values = {key: cell.value for key, cell in cells.items() if cell.value is not None}
fmls = {key: cell.formula for key, cell in cells.items() if cell.formula}

In [ ]:
row = {key: value for key, value in cells.items() if gnc(key) and key.endswith('378')}
{key: row[key] for key in sorted(row.keys(), key=lambda x: fnc(x))}

In [ ]:
ws_name = ws.title
ws_name, full_path, ws_range

In [ ]:
excel_wb = ExcelWorkbook(full_path)
excel_wb.create_worksheet('H4 (Impuesto Diferido)')
wsheet = excel_wb.create_worksheet(ws_name)


In [ ]:
cells = ws[ws_range]
fmls1 = {adr: f"={fml.lstrip('=+')}" for adr, cell in cells.items() if (fml := cell.formula_)}
vals1 = {adr: cell.value for adr, cell in cells.items() if adr not in fmls1}

In [ ]:
# cells = ws_data['cells']
keys = sorted(cells, key=lambda x:fnc(x))
fmls = {key: f"={cells[key][1].lstrip('=+')}" for key in keys if gnc(key) and cells[key][1] is not None}
all_values = {key: cells[key][2] for key in keys if gnc(key) and not isinstance(cells[key][2], str)}


In [ ]:
values = {key: all_values[key] for key in (all_values.keys() - fmls.keys())}


In [ ]:
diff = vals1.keys() - values.keys()
{key: cells[key] for key in diff}


In [ ]:
fmls['K595']

In [ ]:
pythonize_fml(fmls['K595'], table_name='tbl')

In [ ]:
sht_tbl = ExcelTable(wsheet, 'Renta_liquida', ws_range, fmls=fmls, values=values)


In [ ]:
sht_tbl.parent.parameters(M6=49799)

In [ ]:
mtable = sht_tbl.minimun_table(any_row=False)
mtable

In [ ]:
wname = ws.title
ws_path = wb._sheet_names[wname]
h3_xml = wb.zf.load_xmlfile(ws_path, isAbs=False)
h3_xml[:100]

In [ ]:
h3_xml[352862:352898]

In [ ]:
all_values['L418']
[x for x in fmls.keys() if x[1:] == '418']
ws['L418']

In [ ]:
mask = mtable.any(axis=1)
mtable[mask]


In [ ]:
tbl = sht_tbl

In [ ]:
tbl.get_formula('383')

In [ ]:
tbl.set_records({'J383': 683400}, 'value')

In [ ]:
tbl.recalculate(True)

In [ ]:
dmy = tbl.minimun_table()
mask = dmy.any(axis=1)
dmy[mask]

In [ ]:
pythonize_fml(fmls['L386'], table_name='tbl')

In [ ]:
mtable.loc['386']

In [ ]:
sht_tbl.set_records(values, field='value')


In [ ]:
sht_tbl.set_records(fmls, field='fml')

In [ ]:

# Acá se hace una corrección ya que la celda 'G142' es un valor pero se encuentra expresada como formula 
# =151647+1
fmls.pop('G142')
all_values['G142'] = 151_647 + 1


In [ ]:
ws_name = wb.sheetnames[12]
wsheet = excel_wb.create_worksheet(ws_name)
sht_tbl = ExcelTable(wsheet, 'ESF_Patrimonio', 'G11:K227', fmls=fmls, values=values)


### BORRADOR

In [1]:
from xlobjects import XlRef
from excel_workbook import ExcelTable, ExcelWorksheet, ExcelWorkbook, pythonize_fml

In [2]:
wb = ExcelWorkbook('Book')
ws = ExcelWorksheet(wb, 'sht1')

In [4]:
tbl1 = ExcelTable(ws, 'tbl', 'A1:M300', {'A10': '=SUM(B10:C200, Z50)'})

In [5]:
tbl1.data.shape

(384, 6)

In [3]:
tbl2 = ExcelTable(ws, 'tbl', 'A1:M300')
tbl2.set_records({'A10': '=SUM(B10:C200, Z50)'}, field='fml')

In [6]:
tbl2.data.shape

(4, 6)

In [7]:
tbl2.data

,fml,dependents,res_order,ftype,value,code
cell,,,,,,
A10,"=SUM(A3:A2, 'sheet1'!ZZ50)",{},1,$,,A1
C200,,{A1},0,#,,A2
B10,,{A1},0,#,,A3
Z50,,{A1},0,#,0,'sheet1'!ZZ50


In [21]:
from xlobjects import XlErrors

In [34]:
list(XlErrors)[2].code

'ZZZZ2'

In [29]:
XlErrors._member_names_

['REF_ERROR',
 'VALUE_ERROR',
 'DIV_ZERO_ERROR',
 'NAME_ERROR',
 'NUM_ERROR',
 'NULL_ERROR',
 'GETTING_DATA_ERROR',
 'SPILL_ERROR',
 'UNKNOWN_ERROR']

In [41]:
str(list(XlErrors)[3])

'#NAME?'

In [35]:
list(XlErrors)

[<XlErrors.REF_ERROR: '#REF!'>,
 <XlErrors.VALUE_ERROR: '#VALUE!'>,
 <XlErrors.DIV_ZERO_ERROR: '#DIV/0!'>,
 <XlErrors.NAME_ERROR: '#NAME?'>,
 <XlErrors.NUM_ERROR: '#NUM!'>,
 <XlErrors.NULL_ERROR: '#NULL!'>,
 <XlErrors.GETTING_DATA_ERROR: '#GETTING_DATA!'>,
 <XlErrors.SPILL_ERROR: '#SPILL!'>,
 <XlErrors.UNKNOWN_ERROR: '#UNKNOWN!'>]